# 03 — Build Gold Table: Candidate Scoring Summary

Builds the Gold layer from the `candidates` Silver table (which now embeds all feature scores).
Candidates with complete features (original 10) form the **training-eligible** set.
New candidates (C011-C020) have partial features and are included for ML inference.

**Input (Silver):**
- `candidates` — profiles + 8 feature scores + hired label (nulls for new candidates)
- `job_requirements` — 4 open HR requisitions

**Output (Gold):**
- `candidate_scoring_summary` — analysis-ready table with computed total_score

## Setup

Configure catalog/schema widgets and load Silver tables.

In [0]:
dbutils.widgets.text("catalog", "bx4",      "Catalog")
dbutils.widgets.text("schema",  "hrd_2030", "Schema")

catalog = dbutils.widgets.get("catalog")
schema  = dbutils.widgets.get("schema")

print("catalog:", catalog)
print("schema: ", schema)

In [ ]:
from pyspark.sql import functions as F

candidates_df = spark.table(f"{catalog}.{schema}.candidates")
job_req_df    = spark.table(f"{catalog}.{schema}.job_requirements")

total    = candidates_df.count()
complete = candidates_df.filter(F.col("interview_score").isNotNull()).count()  # C001-C010 have full feature scores; C011-C020 are new/unscored
print(f"candidates:       {total} rows total ({complete} with complete features, {total-complete} new/unscored)")
print(f"job_requirements: {job_req_df.count()} rows")

## Build Gold Table

Join candidates with job requirements, compute weighted `total_score`, and write `candidate_scoring_summary` as a Delta table.

In [ ]:
# -----------------------------------------------------------------------
# Build Gold table — join candidates with job_requirements on job_id
# Compute total_score using weighted formula (available features only)
# Weights: Education 10%, Experience 20%, Leadership 20%, Certification 10%,
#          Skills 10%, Industry 10%, Interview 10%, CultureFit 10%
# -----------------------------------------------------------------------
# Rename job_requirements.location before the join to avoid ambiguity
# (both candidates and job_requirements have a 'location' column)
job_req_aliased = job_req_df.withColumnRenamed("location", "job_location")

gold_df = (
    candidates_df
    .join(job_req_aliased, "job_id", "left")
    .withColumn(
        "total_score",
        F.when(
            F.col("interview_score").isNotNull(),
            F.round(
                F.col("education_score")          * 0.10 +
                F.col("experience_score")         * 0.20 +
                F.col("leadership_score")         * 0.20 +
                F.col("certification_score")      * 0.10 +
                F.col("skills_match_score")       * 0.10 +
                F.col("industry_relevance_score") * 0.10 +
                F.col("interview_score")          * 0.10 +
                F.col("culture_fit")              * 0.10,
                1
            )
        ).otherwise(F.lit(None))
    )
    .select(
        "candidate_id", "first_name", "last_name", "email",
        "current_title", "current_company",
        "years_total_experience", "years_hr_experience", "years_leadership",
        "education_level", "certifications", "industries",
        "direct_reports_managed", "budget_managed_millions", "location", "resume_filename",
        "job_id",
        "education_score", "experience_score", "leadership_score",
        "certification_score", "skills_match_score", "industry_relevance_score",
        "interview_score", "culture_fit", "total_score", "hired",
        F.col("title").alias("job_title"),
        "department",
        F.col("job_location"),
        "salary_min", "salary_max",
    )
)

gold_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.{schema}.candidate_scoring_summary")
count = spark.table(f"{catalog}.{schema}.candidate_scoring_summary").count()
print(f"✓ candidate_scoring_summary created — {count} rows")

In [ ]:
# -----------------------------------------------------------------------
# Add Gold table comment
# -----------------------------------------------------------------------
spark.sql(f"""
COMMENT ON TABLE {catalog}.{schema}.candidate_scoring_summary IS
  'Gold layer: Joined candidate profiles with hiring scores and job requirements for 4 open HR roles.
Denormalized from Silver tables (candidates, job_requirements) for analytical and AI use.
Contains 20 candidates (C001-C020) across JR001-JR004. Rows with NULL total_score are new candidates
awaiting interview scoring. Used for ML training, Genie analytics, and agent-based hiring recommendations
in the Jackson and Jackson HR Digital initiative. Primary key: candidate_id. Sort by total_score DESC to rank candidates.'
""")
print("Gold table comment set.")

## Verify Gold Table

Display the full gold table and confirm row counts, score distributions, and new candidates.

In [0]:
# -----------------------------------------------------------------------
# Display the full Gold table
# -----------------------------------------------------------------------
gold_display = spark.table(f"{catalog}.{schema}.candidate_scoring_summary") \
    .orderBy("total_score", ascending=False)

display(gold_display)

In [ ]:
gold = spark.table(f"{catalog}.{schema}.candidate_scoring_summary")

print("--- All 20 Candidates ---")
gold.select("candidate_id", "first_name", "last_name", "job_id", "total_score", "hired") \
    .orderBy(F.col("total_score").desc_nulls_last()) \
    .show(20, truncate=False)

print("--- Training-eligible candidates (complete features) ---")
gold.filter(F.col("interview_score").isNotNull()) \
    .groupBy("hired") \
    .agg(F.round(F.avg("total_score"), 1).alias("avg_score"), F.count("*").alias("count")) \
    .orderBy("hired", ascending=False) \
    .show()

print("--- New candidates awaiting prediction ---")
gold.filter(F.col("interview_score").isNull()) \
    .select("candidate_id", "first_name", "last_name", "job_id", "education_score", "experience_score") \
    .show(truncate=False)

## Next Steps

The Gold table `candidate_scoring_summary` is now ready for:

| Step | Notebook | Description |
|------|----------|-------------|
| Business Semantics | `03_1_apply_business_semantics` | Create metric views on Gold for governed analytics |
| Genie Space | `04_create_genie_space` | Wire the Gold table + metric views into a Genie Space |
| Agent | `hire_right_agent.py` | Compound AI agent using Gold + vector search for recommendations |
| App | `app.py` | Databricks App exposing the agent via a hiring dashboard UI |

> **Tip:** Run `03_1_apply_business_semantics` next to add the `hiring_analytics` metric view.